# Backpropagation & Computational Graphs
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> Chain rule of calculus, Reverse-mode automatic differentiation, Jacobian matrices, Vanishing/Exploding gradients, and Weight Initialization.

---

### 📖 Conceptual Foundations

#### What is Backpropagation?

Introduced to machine learning by Rumelhart, Hinton, and Williams in 1986, **Backpropagation (Backward Propagation of Errors)** is an algorithm for computing the gradient of the loss function with respect to every trainable weight and bias in a neural network by recursively applying the **multivariate chain rule** of calculus in reverse topological order over a computational graph.


#### Forward Pass vs Backward Pass on a Computational Graph

Every deep learning framework (PyTorch, TensorFlow, JAX) represents calculations as a Directed Acyclic Graph (DAG) of atomic operations:

1. **Forward Pass:** Compute intermediate values from inputs to loss, storing intermediate activations ($Z^{[l]}, A^{[l]}$) in memory (activations cache).
2. **Backward Pass (Reverse-Mode Autodiff):** Traverse backward from the scalar loss $L$, computing the gradient of $L$ with respect to each node by multiplying upstream gradients with local Jacobians:

$$\frac{\partial L}{\partial x} = \sum_{y \in \text{children}(x)} \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}$$


#### The Core Backpropagation Equations for an MLP Layer

For layer $l$ where $Z^{[l]} = W^{[l]} A^{[l-1]} + \mathbf{b}^{[l]}$ and $A^{[l]} = g^{[l]}(Z^{[l]})$:

1. **Error vector at layer $l$:**
$$dZ^{[l]} = \frac{\partial L}{\partial Z^{[l]}} = dA^{[l]} \odot {g^{[l]}}'(Z^{[l]})$$

2. **Gradient with respect to weights:**
$$dW^{[l]} = \frac{\partial L}{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} (A^{[l-1]})^T$$

3. **Gradient with respect to biases:**
$$d\mathbf{b}^{[l]} = \frac{\partial L}{\partial \mathbf{b}^{[l]}} = \frac{1}{m} \sum_{i=1}^{m} dZ^{[l](i)}$$

4. **Upstream gradient for previous layer:**
$$dA^{[l-1]} = (W^{[l]})^T dZ^{[l]}$$


#### The Two Pathologies: Vanishing vs Exploding Gradients

| Pathology | Mathematical Cause | Observed Symptoms in Training | Proven Remedies |
| --- | --- | --- | --- |
| Vanishing Gradients | Repeated multiplication of small weight matrices ($\lambda_{max} < 1$) and saturating derivatives (Sigmoid $\le 0.25$); $\prod_{l=1}^L W^{[l]} \to 0$. | Early layers learn exceedingly slowly or completely freeze; training loss plateaus early. | Use ReLU / GELU activations; He/Xavier initialization; Residual connections (ResNets); Batch/Layer Normalization. |
| Exploding Gradients | Repeated multiplication of large weight matrices ($\lambda_{max} > 1$); $\prod_{l=1}^L W^{[l]} \to \infty$. | Weights oscillate wildly; loss abruptly evaluates to NaN or Inf; model diverges. | Gradient Clipping (by norm or value); proper weight initialization; lower learning rate; Weight Decay. |


#### Gradient Clipping Techniques

**Norm Clipping** scales down the entire gradient vector if its L2 norm exceeds a threshold $c$ without changing its direction:

$$\mathbf{g} \leftarrow \mathbf{g} \cdot \min\left(1, \frac{c}{\|\mathbf{g}\|_2}\right)$$

This is essential when training RNNs, LSTMs, and large Transformer models to prevent catastrophic divergence from occasional massive loss spikes.



In [ ]:
# %matplotlib inline
import numpy as np

# Analytical vs Numerical gradient checker
def numerical_gradient(f, w, eps=1e-5):
    grad = np.zeros_like(w)
    it = np.nditer(w, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        orig_val = w[idx]
        
        w[idx] = orig_val + eps
        loss_pos = f(w)
        w[idx] = orig_val - eps
        loss_neg = f(w)
        w[idx] = orig_val # restore
        
        grad[idx] = (loss_pos - loss_neg) / (2.0 * eps)
        it.iternext()
    return grad

# Simple 2-layer forward and loss
np.random.seed(42)
X = np.random.randn(3, 5)
Y = np.array([[1, 0, 1, 0, 1]])
W1 = np.random.randn(4, 3) * 0.1

def loss_fn(W1_val):
    Z1 = np.dot(W1_val, X)
    A1 = np.maximum(0, Z1) # ReLU
    # Mock output loss
    return 0.5 * np.sum(A1**2)

# Analytical gradient for W1
Z1 = np.dot(W1, X)
A1 = np.maximum(0, Z1)
dA1 = A1
dZ1 = dA1 * (Z1 > 0)
analytical_grad = np.dot(dZ1, X.T)

num_grad = numerical_gradient(loss_fn, W1)
relative_error = np.linalg.norm(analytical_grad - num_grad) / (np.linalg.norm(analytical_grad) + np.linalg.norm(num_grad))

print(f'Analytical Gradient Shape: {analytical_grad.shape}')
print(f'Numerical Gradient Shape:  {num_grad.shape}')
print(f'Relative Error: {relative_error:.2e}')
assert relative_error < 1e-7, 'Gradient check failed!'


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

